## Single Head Attention using Pytorch .

## Complete FLow with Mathametical Explanation :

For $node (i)$ :

$x_i \xrightarrow{W} h_i \quad (h_i = Wx_i)$

For every $neighbor (j)$ :

$[h_i || h_j] \xrightarrow{a^T} e_{ij} \quad (e_{ij} = \text{LeakyReLU}(a^T[h_i || h_j]))$

Normalize :

$e_{ij} \xrightarrow{Softmax} \alpha_{ij} \quad (\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}_i} \exp(e_{ik})})$

Aggregate :

$h_i' = \sum_{j \in \mathcal{N}_i} \alpha_{ij} h_j$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def _sparse_softmax(e: torch.tensor , dst : torch.tensor , N : int ):
  """
  e: Attention scores  has dimension (E,) E is the number of nodes .
  dst : Destination nodes of the edges has dimension (E,) E is the number of nodes .
  N: Number of nodes
  """
  e_max = torch.full((N,),float('-inf')) #[N,]
  e_max.scatter_reduce_(0 , dst , e , reduce = 'amax', include_self = True)
  e = torch.exp(e - e_max[dst])

  denom = torch.zeros(N)
  denom.scatter_add_(0,dst,e)

  return e/(denom[dst] + 1e-16)

In [ ]:
class GAT_Single_Head(nn.Module):
  def __init__(self, in_features: int , out_features: int , dropout:float = 0.2 , alpha:float = 0.2):
    super().__init__()
    self.in_features = in_features
    self.out_features = out_features
    self.dropout = dropout
    self.alpha = alpha

    self.W = nn.Linear(in_features , out_features)
    self.a_src = nn.Parameter(torch.empty(out_features,1))
    self.a_dst = nn.Parameter (torch.empty(out_features,1))

    self.leakyRELU = nn.LeakyReLU(self.alpha)
    self._init_weights()

  def _init_weights(self):
    nn.init.xavier_uniform(self.W.weight)
    nn.init.xavier_uniform(self.a_src)
    nn.init.xavier_uniform(self.a_dst)

  def forward(self, x: torch.tensor , edge_index: torch.tensor ):
    h = self.W(x) #[N , out_features]
    h = F.dropout(h, self.dropout, training=self.training)
    src , dst = edge_index[0] , edge_index[1]  # each [E,]

    e = self.leakyRELU((h @ self.a_src)[src] + (h @ self.a_dst)[dst]) #[E,]

    alpha = _sparse_softmax(e.squeeze(-1),dst,x.shape[0]) #[E,]
    alpha = F.dropout(alpha, self.dropout, training=self.training)

    out = torch.zeros_like(h)
    out.scatter_add_(0, dst.unsqueeze(-1).expand(-1 , self.out_features) , alpha.unsqueeze(-1) * h[src])
    return F.elu(out)



In [ ]:
def execution():
  print("\n" + "="*60)
  print("Single-Head GAT Layer (PyTorch)")
  print("="*60)
  N, F_in, F_out = 5, 8, 4
  x = torch.randn(N, F_in)
  # Simple path graph: 0→1→2→3→4 with self-loops
  edges = [[0,1,2,3,4,0,1,2,3,4],[0,1,2,3,4,1,2,3,4,0]]
  edge_index = torch.tensor(edges, dtype=torch.long)

  layer = GAT_Single_Head(F_in, F_out)
  out = layer(x, edge_index)
  print(f"Input  : {x.shape}")
  print(f"Output : {out.shape}")
  print(f"Sample output (node 0): {out[0].detach().numpy().round(4)}")
  print(f"Sample output (node 1): {out[1].detach().numpy().round(4)}")
  print(f"Sample output (node 2): {out[2].detach().numpy().round(4)}")
  print(f"Sample output (node 3): {out[3].detach().numpy().round(4)}")
  print(f"Sample output (node 4): {out[4].detach().numpy().round(4)}")
  print("\n" + "="*60)

In [ ]:
execution()


Single-Head GAT Layer (PyTorch)
Input  : torch.Size([5, 8])
Output : torch.Size([5, 4])
Sample output (node 0): [-0.2678  1.7343  0.     -0.2137]
Sample output (node 1): [ 1.8735 -0.2271 -0.0547 -0.7656]
Sample output (node 2): [ 2.5742  0.2194 -0.8694 -0.7532]
Sample output (node 3): [-0.2058  1.0926  0.1004 -0.3215]
Sample output (node 4): [-0.6726  1.0178  0.1538 -0.4774]



/tmp/ipykernel_6702/1447937430.py:17: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.W.weight)
/tmp/ipykernel_6702/1447937430.py:18: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.a_src)
/tmp/ipykernel_6702/1447937430.py:19: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.a_dst)


## GAT of Multi head Attention using Pytorch .

After the loop finishes, you have $K$ different representations for every single node. The GAT paper handles combining them in two distinct ways, governed by your self.concat flag:Mode A: Concatenation (concat=True)Used for all hidden layers in the network.If you have 8 heads and out_features is 64, the model concatenates them side-by-side.torch.cat(head_outputs, dim=-1) results in a massive new feature vector for each node of size $8 \times 64 = 512$.$$h_i' = \text{ELU}\left(\Big\Vert_{k=1}^{K} \sum_{j \in \mathcal{N}(i)} \alpha_{ij}^k W_k h_j\right)$$(where $\Vert$ denotes concatenation)Mode B: Averaging (concat=False)Used for the final output layer (e.g., right before a softmax for classification).Concatenating at the final layer would change the desired output dimension (e.g., the number of classes). Instead, torch.stack(head_outputs).mean(0) stacks the outputs and calculates the mean across the heads, keeping the final dimension strictly at out_features.$$h_i' = \text{ELU}\left(\frac{1}{K} \sum_{k=1}^{K} \sum_{j \in \mathcal{N}(i)} \alpha_{ij}^k W_k h_j\right)$$Finally, you apply the ELU non-linearity to the combined output. The implementation is structurally sound and mathematically accurate to the original GAT architecture.

In [ ]:
class GAT_Multihead_Attention(nn.Module):
  def __init__(self, in_features : int , out_features : int , concat : bool = True , n_heads : int = 6 , dropout: float = 0.2 , alpha : float = 0.2 ):
    super().__init__()
    self.in_features = in_features
    self.out_features = out_features
    self.concat = concat
    self.n_heads = n_heads
    self.dropout = dropout
    self.alpha = alpha

    self.W = nn.ModuleList([nn.Linear(in_features , out_features , bias = False) for _ in range(n_heads)])
    self.a_src = nn.ParameterList([nn.Parameter(torch.empty(out_features,1)) for _ in range(n_heads)])
    self.a_dst = nn.ParameterList([nn.Parameter(torch.empty(out_features,1)) for _ in range(n_heads)])

    self.leakyRELU = nn.LeakyReLU(alpha)
    self._init_weights()

  def _init_weights(self):
    for i in range(self.n_heads):
      nn.init.xavier_uniform(self.W[i].weight)
      nn.init.xavier_uniform(self.a_src[i])
      nn.init.xavier_uniform(self.a_dst[i])

  def forward(self, x: torch.tensor , edge_index: torch.tensor ):
    src , dst = edge_index[0] , edge_index[1]  # each [E,]
    N = x.shape[0]
    head_outputs = []

    for k in range(self.n_heads):
      h = self.W[k](x) #[N , out_features]
      h = F.dropout(h, self.dropout, training=self.training)

      e = self.leakyRELU((h @ self.a_src[k])[src] + (h @ self.a_dst[k])[dst]) #[E,]

      alpha = _sparse_softmax(e.squeeze(-1), dst , N) # [E,]
      alpha = F.dropout(alpha, self.dropout, training=self.training)

      out_k = torch.zeros_like(h)
      out_k.scatter_add_(0, dst.unsqueeze(-1).expand(-1, self.out_features)  , alpha.unsqueeze(-1) * h[src])

      head_outputs.append(out_k)

    if self.concat:
      out = torch.cat(head_outputs , dim = -1) # [N , out_features * n_heads]
    else:
      out = torch.stack(head_outputs).mean(dim = 0 ) # [N , out_features]

    return F.elu(out)



In [ ]:
def execute_GAT_MHA():
    print("\n" + "="*60)
    print("LEVEL 3: Multi-Head GAT Layer (PyTorch)")
    print("="*60)
    N, F_in = 6, 16
    K = 4   # heads
    x = torch.randn(N, F_in)
    edge_index = torch.tensor([
        [0,1,2,3,4,5,0,2,4],
        [1,2,3,4,5,0,3,5,1]
    ], dtype=torch.long)

    layer = GAT_Multihead_Attention(F_in, out_features=8, n_heads=K, concat=True)
    out = layer(x, edge_index)
    print(f"Input  : {x.shape}")
    print(f"Output (concat): {out.shape}  — expected [{N}, {K*8}]")
    print(f"Sample output (node 0): {out[0].detach().numpy().round(4)}")
    print(f"Sample output (node 1): {out[1].detach().numpy().round(4)}")
    print("\n" + "="*60)

    layer_avg = GAT_Multihead_Attention(F_in, out_features=8, n_heads=K, concat=False)
    out_avg = layer_avg(x, edge_index)
    print(f"Output (avg)   : {out_avg.shape}")
    print(f"Sample output (node 0): {out_avg[0].detach().numpy().round(4)}")
    print(f"Sample output (node 1): {out_avg[1].detach().numpy().round(4)}")
    print("\n" + "="*60)


In [ ]:
execute_GAT_MHA()


LEVEL 3: Multi-Head GAT Layer (PyTorch)
Input  : torch.Size([6, 16])
Output (concat): torch.Size([6, 32])  — expected [6, 32]
Sample output (node 0): [-0.666   0.      0.      1.6263  0.      0.1799 -0.2156  1.0397  0.
  0.      0.      0.      0.      0.      0.      0.     -0.854   4.2384
  0.3369 -0.9808 -0.3791 -0.7821  0.     -0.9176  0.      1.988   0.7002
  0.     -0.9341 -0.9917 -0.956   4.5538]
Sample output (node 1): [-0.8059 -0.8371  1.5089  0.1233  2.661  -0.2337 -0.0533 -0.4646 -0.8356
  0.0637  0.0529  0.3038  0.5605 -0.0174  0.6989 -0.8335 -0.092  -0.0407
 -0.0106 -0.1601  0.0475  0.2805 -0.1572 -0.1982  1.0444 -0.3118  0.1527
 -0.7537 -0.3861 -0.586  -0.4645  0.    ]

Output (avg)   : torch.Size([6, 8])
Sample output (node 0): [ 0.1517  0.2382 -0.4272 -0.6775 -0.3563 -0.7786  0.1591  0.7042]
Sample output (node 1): [ 0.3265  0.2595 -0.5351 -0.5392  0.2236  0.4905  0.1585  0.9065]



/tmp/ipykernel_6702/3153466575.py:20: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.W[i].weight)
/tmp/ipykernel_6702/3153466575.py:21: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.a_src[i])
/tmp/ipykernel_6702/3153466575.py:22: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.a_dst[i])


In [ ]:
class Full_GAT(nn.Module):
  def __init__(self, in_features : int , hidden: int , n_classes : int , dropout : float = 0.2 ):
    super().__init__()
    self.in_features = in_features
    self.hidden = hidden
    self.n_classes = n_classes
    self.dropout = dropout

    K1 , K2 = 8, 1

    self.gat1 = GAT_Multihead_Attention(in_features , hidden , concat = True , n_heads= K1 , dropout=dropout)
    self.ln1 = nn.LayerNorm(K1*hidden)

    if self.in_features != K1* hidden :
      self.skip1 = nn.Linear(in_features, K1*hidden)
    else:
      self.skip1 = nn.Indentity()

    self.gat2 = GAT_Multihead_Attention(K1*hidden , n_classes , concat = False , n_heads= K2 , dropout=dropout)
    self.ln2 = nn.LayerNorm(n_classes)

  def forward(self , x: torch.tensor , edge_index : torch.tensor ):
    x = F.dropout(x, self.dropout, self.training)
    h1 = self.gat1(x, edge_index)
    h1 = self.ln1(h1 + self.skip1(x))

    h2 = self.gat2(h1, edge_index)
    h2 = self.ln2(h2)

    return F.log_softmax(h2, dim=1)

In [ ]:
def execute_Full_GAT():
    print("\n" + "="*60)
    print("Full GAT Model (2-layer + skip + LayerNorm)")
    print("="*60)
    N, F_in, n_classes = 100, 1433, 7
    x = torch.randn(N, F_in)
    # Random sparse graph
    src = torch.randint(0, N, (300,))
    dst = torch.randint(0, N, (300,))
    edge_index = torch.stack([src, dst])

    model = Full_GAT(F_in, hidden=8, n_classes=n_classes, dropout=0.6)
    model.eval()
    with torch.no_grad():
        out = model(x, edge_index)
    print(f"Input  : {x.shape}")
    print(f"Output : {out.shape}  (log-softmax probabilities)")
    print(f"Params : {sum(p.numel() for p in model.parameters()):,}")
    print(f"Sample output (node 0): {out[0].detach().numpy().round(4)}")
    print(f"Sample output (node 1): {out[1].detach().numpy().round(4)}")
    print(f"Sample output (node 2): {out[2].detach().numpy().round(4)}")
    print("\n" + "="*60)


In [ ]:
execute_Full_GAT()


Full GAT Model (2-layer + skip + LayerNorm)
Input  : torch.Size([100, 1433])
Output : torch.Size([100, 7])  (log-softmax probabilities)
Params : 184,220
Sample output (node 0): [-3.2431 -3.2675 -2.0861 -0.9036 -3.4634 -1.1452 -3.1212]
Sample output (node 1): [-3.834  -1.5519 -2.3637 -0.6839 -3.4722 -2.6794 -2.6861]
Sample output (node 2): [-2.612  -4.1051 -1.2555 -2.8221 -3.0053 -1.857  -1.0215]



/tmp/ipykernel_6702/3153466575.py:20: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.W[i].weight)
/tmp/ipykernel_6702/3153466575.py:21: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.a_src[i])
/tmp/ipykernel_6702/3153466575.py:22: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  nn.init.xavier_uniform(self.a_dst[i])
